# Portfolio Optimisation

## Objective

This notebook investigates whether formal portfolio optimisation improves the allocation between the momentum and realised-volatility sleeves.

The analysis progresses from relatively robust, risk-based methods to more assumption-sensitive return-based methods:

1. global minimum variance;
2. maximum diversification;
3. maximum Sharpe;
4. regularised and turnover-aware optimisation.

All methods are implemented walk-forward. Parameters are estimated using only information available before each allocation decision, and optimised portfolios are compared with the existing allocation baselines and SPY.

## Design principles

- Optimise between factor sleeves rather than individual securities.
- Estimate allocation parameters from historical sleeve returns.
- Apply each estimated allocation only to subsequent returns.
- Use long-only sleeve weights that sum to one.
- Reconstruct combined security-level holdings before calculating turnover and   transaction costs.
- Compare optimisation methods over an identical evaluation period.
- Treat robustness across windows and subperiods as more important than the   best full-sample result.

### Load previous portfolio results

In [1]:
import numpy as np
import pandas as pd

from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet


SLEEVE_NAMES = [
    "Momentum",
    "Realised Volatility",
]

TRANSACTION_COST_BPS = 10.0

# Daily pre-cost sleeve returns used for estimation.

sleeve_return_frame = (
    load_parquet(PROCESSED_DATA_DIR / "portfolio_optimisation_sleeve_returns.parquet")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")
    .sort_index()
)

sleeve_return_frame = sleeve_return_frame[SLEEVE_NAMES]


# Stock-level sleeve targets used for backtesting.

sleeve_target_weights = load_parquet(
    PROCESSED_DATA_DIR / "portfolio_optimisation_sleeve_targets.parquet"
)

sleeve_target_weights["date"] = pd.to_datetime(sleeve_target_weights["date"])

sleeve_targets = {
    sleeve_name: (
        sleeve_target_weights.loc[
            sleeve_target_weights["sleeve"].eq(sleeve_name),
            ["date", "ticker", "weight"],
        ]
        .sort_values(["date", "ticker"])
        .reset_index(drop=True)
    )
    for sleeve_name in SLEEVE_NAMES
}


# Existing portfolio and SPY benchmarks.

benchmark_daily = load_parquet(
    PROCESSED_DATA_DIR / "portfolio_optimisation_benchmarks.parquet"
)

benchmark_daily["date"] = pd.to_datetime(benchmark_daily["date"])

benchmark_portfolios = {
    portfolio_name: (
        portfolio_data.drop(columns="portfolio")
        .sort_values("date")
        .reset_index(drop=True)
    )
    for portfolio_name, portfolio_data in benchmark_daily.groupby(
        "portfolio",
        sort=False,
    )
}

In [2]:
factor_panel = load_parquet(PROCESSED_DATA_DIR / "factor_panel.parquet")

return_panel = (
    factor_panel[["date", "ticker", "forward_ret_1d"]]
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

## 1. Global minimum-variance allocation

For sleeve covariance matrix

$$
\Sigma_t =
\begin{pmatrix}
\sigma_{\mathrm{mom}}^2 & \sigma_{\mathrm{mom,vol}} \\
\sigma_{\mathrm{mom,vol}} & \sigma_{\mathrm{vol}}^2
\end{pmatrix},
$$

the momentum weight that minimises portfolio variance is

$$
w_{\mathrm{mom},t}^*
=
\frac{
\sigma_{\mathrm{vol}}^2-\sigma_{\mathrm{mom,vol}}
}{
\sigma_{\mathrm{mom}}^2+\sigma_{\mathrm{vol}}^2
-2\sigma_{\mathrm{mom,vol}}
}.
$$

The realised-volatility sleeve receives the remaining weight:

$$
w_{\mathrm{vol},t}^*=1-w_{\mathrm{mom},t}^*.
$$

The analytical solution is clipped to the interval [0,1] to impose long-only sleeve weights.

At each rebalance date, the covariance matrix is estimated using at most the preceding 252 daily sleeve returns. A minimum of 126 observations is required. Returns dated on or after the allocation date are excluded.

In [4]:
GMV_LOOKBACK_DAYS = 252
GMV_MIN_OBSERVATIONS = 126

MOMENTUM_SLEEVE = "Momentum"
VOLATILITY_SLEEVE = "Realised Volatility"

In [6]:
# Derive the common rebalance dates from the two sleeve target panels.

momentum_rebalance_dates = pd.DatetimeIndex(
    pd.to_datetime(sleeve_targets[MOMENTUM_SLEEVE]["date"].unique())
)

volatility_rebalance_dates = pd.DatetimeIndex(
    pd.to_datetime(sleeve_targets[VOLATILITY_SLEEVE]["date"].unique())
)

if set(momentum_rebalance_dates) != set(volatility_rebalance_dates):
    raise ValueError(
        "Momentum and realised-volatility sleeves "
        "have different rebalance-date sets."
    )

rebalance_dates = pd.DatetimeIndex(sorted(momentum_rebalance_dates))

# Exclude target dates outside the available return history.
rebalance_dates = rebalance_dates[
    (rebalance_dates >= sleeve_return_frame.index.min())
    & (rebalance_dates <= sleeve_return_frame.index.max())
]

print(
    "Rebalance period:",
    rebalance_dates.min().date(),
    "to",
    rebalance_dates.max().date(),
)

print("Number of rebalance dates:", len(rebalance_dates))

Rebalance period: 2016-01-07 to 2026-06-25
Number of rebalance dates: 527


#### Analytical weight calculation

In [5]:
def calculate_two_sleeve_gmv_weights(
    covariance_matrix,
    lower_bound=0.0,
    upper_bound=1.0,
):
    """
    Calculate analytical two-asset global minimum-variance weights.

    Returns both the unconstrained momentum weight and the
    long-only constrained sleeve weights.
    """
    momentum_variance = covariance_matrix.loc[
        MOMENTUM_SLEEVE,
        MOMENTUM_SLEEVE,
    ]

    volatility_variance = covariance_matrix.loc[
        VOLATILITY_SLEEVE,
        VOLATILITY_SLEEVE,
    ]

    sleeve_covariance = covariance_matrix.loc[
        MOMENTUM_SLEEVE,
        VOLATILITY_SLEEVE,
    ]

    denominator = (
        momentum_variance
        + volatility_variance
        - 2.0 * sleeve_covariance
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 1e-14
    ):
        unconstrained_momentum_weight = 0.5
    else:
        unconstrained_momentum_weight = (
            volatility_variance
            - sleeve_covariance
        ) / denominator

    momentum_weight = float(
        np.clip(
            unconstrained_momentum_weight,
            lower_bound,
            upper_bound,
        )
    )

    volatility_weight = 1.0 - momentum_weight

    weights = pd.Series(
        {
            MOMENTUM_SLEEVE: momentum_weight,
            VOLATILITY_SLEEVE: volatility_weight,
        },
        dtype=float,
    )

    return {
        "weights": weights,
        "unconstrained_momentum_weight": float(
            unconstrained_momentum_weight
        ),
        "denominator": float(denominator),
    }

#### Estimate the weights walk-forward

In [7]:
gmv_allocation_records = []

for allocation_date in rebalance_dates:
    # Strictly exclude the return dated on the allocation date.
    available_history = (
        sleeve_return_frame.loc[
            sleeve_return_frame.index < allocation_date,
            SLEEVE_NAMES,
        ]
        .dropna(how="any")
        .tail(GMV_LOOKBACK_DAYS)
    )

    if len(available_history) < GMV_MIN_OBSERVATIONS:
        continue

    estimation_history = available_history.copy()

    covariance_matrix = estimation_history.cov(
        min_periods=GMV_MIN_OBSERVATIONS,
    )

    correlation_matrix = estimation_history.corr(
        method="pearson",
        min_periods=GMV_MIN_OBSERVATIONS,
    )

    if covariance_matrix.isna().any().any() or correlation_matrix.isna().any().any():
        raise ValueError(
            f"Invalid covariance or correlation estimate "
            f"for {allocation_date.date()}."
        )

    solution = calculate_two_sleeve_gmv_weights(covariance_matrix)

    weights = solution["weights"]

    gmv_allocation_records.append(
        {
            "date": allocation_date,
            "estimation_start": estimation_history.index.min(),
            "estimation_end": estimation_history.index.max(),
            "observations": len(estimation_history),
            "momentum_weight": weights[MOMENTUM_SLEEVE],
            "realised_volatility_weight": weights[VOLATILITY_SLEEVE],
            "unconstrained_momentum_weight": solution["unconstrained_momentum_weight"],
            "momentum_volatility": np.sqrt(
                covariance_matrix.loc[
                    MOMENTUM_SLEEVE,
                    MOMENTUM_SLEEVE,
                ]
                * 252
            ),
            "realised_volatility_volatility": np.sqrt(
                covariance_matrix.loc[
                    VOLATILITY_SLEEVE,
                    VOLATILITY_SLEEVE,
                ]
                * 252
            ),
            "sleeve_correlation": correlation_matrix.loc[
                MOMENTUM_SLEEVE,
                VOLATILITY_SLEEVE,
            ],
        }
    )

gmv_allocations = (
    pd.DataFrame(gmv_allocation_records).sort_values("date").reset_index(drop=True)
)

if gmv_allocations.empty:
    raise ValueError(
        "No rebalance date has enough historical " "observations for GMV estimation."
    )

display(gmv_allocations.head())
display(gmv_allocations.tail())

,date,estimation_start,estimation_end,observations,momentum_weight,realised_volatility_weight,unconstrained_momentum_weight,momentum_volatility,realised_volatility_volatility,sleeve_correlation
0,2016-07-14,2016-01-07,2016-07-13,130,0.581028,0.418972,0.581028,0.177978,0.221748,-0.343491
1,2016-07-21,2016-01-07,2016-07-20,135,0.585100,0.414900,0.585100,0.174943,0.220236,-0.338116
2,2016-07-28,2016-01-07,2016-07-27,140,0.588479,0.411521,0.588479,0.172890,0.219300,-0.328060
3,2016-08-04,2016-01-07,2016-08-03,145,0.589301,0.410699,0.589301,0.170698,0.217149,-0.331557
4,2016-08-11,2016-01-07,2016-08-10,150,0.588365,0.411635,0.588365,0.170077,0.216477,-0.349019


,date,estimation_start,estimation_end,observations,momentum_weight,realised_volatility_weight,unconstrained_momentum_weight,momentum_volatility,realised_volatility_volatility,sleeve_correlation
496,2026-05-27,2025-05-23,2026-05-26,252,0.437715,0.562285,0.437715,0.255474,0.239132,0.471133
497,2026-06-03,2025-06-02,2026-06-02,252,0.430737,0.569263,0.430737,0.261954,0.241823,0.425338
498,2026-06-10,2025-06-09,2026-06-09,252,0.470844,0.529156,0.470844,0.266385,0.258246,0.468262
499,2026-06-17,2025-06-16,2026-06-16,252,0.454680,0.545320,0.454680,0.274545,0.262156,0.491437
500,2026-06-25,2025-06-24,2026-06-24,252,0.436779,0.563221,0.436779,0.284628,0.267893,0.522299


#### Validate the walk-forward construction

In [8]:
weight_sums = (
    gmv_allocations["momentum_weight"] + gmv_allocations["realised_volatility_weight"]
)

if not np.allclose(weight_sums, 1.0):
    raise ValueError("GMV sleeve weights do not sum to one.")

if (
    not gmv_allocations[
        [
            "momentum_weight",
            "realised_volatility_weight",
        ]
    ]
    .ge(0.0)
    .all()
    .all()
):
    raise ValueError("Negative long-only GMV weights found.")

if (
    not gmv_allocations[
        [
            "momentum_weight",
            "realised_volatility_weight",
        ]
    ]
    .le(1.0)
    .all()
    .all()
):
    raise ValueError("GMV weights above one found.")

if not (gmv_allocations["estimation_end"] < gmv_allocations["date"]).all():
    raise ValueError("Look-ahead detected in the estimation windows.")

if gmv_allocations["date"].duplicated().any():
    raise ValueError("Duplicate GMV allocation dates found.")

print(
    "Allocation period:",
    gmv_allocations["date"].min().date(),
    "to",
    gmv_allocations["date"].max().date(),
)

print(
    "Number of allocations:",
    len(gmv_allocations),
)

print(
    "Full-window allocations:",
    gmv_allocations["observations"].eq(GMV_LOOKBACK_DAYS).sum(),
)

print(
    "Boundary solutions:",
    gmv_allocations["momentum_weight"].isin([0.0, 1.0]).sum(),
)

Allocation period: 2016-07-14 to 2026-06-25
Number of allocations: 501
Full-window allocations: 476
Boundary solutions: 5


In [9]:
gmv_weight_summary = gmv_allocations[
    [
        "momentum_weight",
        "realised_volatility_weight",
        "unconstrained_momentum_weight",
        "momentum_volatility",
        "realised_volatility_volatility",
        "sleeve_correlation",
    ]
].describe().T

display(gmv_weight_summary)

,count,mean,std,min,25%,50%,75%,max
momentum_weight,501.0,0.542800,0.191067,0.000000,0.465240,0.564023,0.695123,0.911079
realised_volatility_weight,501.0,0.457200,0.191067,0.088921,0.304877,0.435977,0.534760,1.000000
unconstrained_momentum_weight,501.0,0.542433,0.192156,-0.067101,0.465240,0.564023,0.695123,0.911079
momentum_volatility,501.0,0.204743,0.058404,0.130390,0.158702,0.193512,0.235500,0.372699
realised_volatility_volatility,501.0,0.234909,0.068952,0.109785,0.189749,0.222076,0.288230,0.374450
sleeve_correlation,501.0,0.103299,0.511675,-0.875279,-0.349567,0.193597,0.595001,0.797763


In [10]:
constraint_diagnostics = pd.Series(
    {
        "unconstrained_below_zero": (
            gmv_allocations["unconstrained_momentum_weight"] < 0.0
        ).sum(),
        "unconstrained_above_one": (
            gmv_allocations["unconstrained_momentum_weight"] > 1.0
        ).sum(),
        "constrained_fraction": (
            (gmv_allocations["unconstrained_momentum_weight"] < 0.0)
            | (gmv_allocations["unconstrained_momentum_weight"] > 1.0)
        ).mean(),
    }
)

display(constraint_diagnostics.to_frame("value"))

,value
unconstrained_below_zero,5.00000
unconstrained_above_one,0.00000
constrained_fraction,0.00998


### GMV allocation diagnostics

The walk-forward GMV portfolio begins in July 2016 after accumulating the required minimum return history. Of the 501 allocations, 476 use the full 252-day estimation window.

The optimiser assigns an average weight of 54.3% to momentum and 45.7% to realised volatility. Momentum receives the larger average allocation partly because its estimated standalone volatility is lower: 20.5% annually, compared with 23.5% for the realised-volatility sleeve.

The solution is generally well behaved. Only five allocations, approximately 1% of the sample, are affected by the long-only constraint. In each case the unconstrained momentum weight is slightly negative and is therefore clipped to zero. No unconstrained estimate exceeds one.

Estimated sleeve correlation is highly time-varying, ranging from -0.88 to 0.80. Consequently, the momentum allocation also varies materially over time, with a standard deviation of 19.1%. This variation may allow the optimiser to respond to changing diversification conditions, but it also introduces estimation risk that will later be addressed through regularisation.

### Backtest

In [11]:
from alpha_research.backtest import (
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.portfolio import (
    combine_dynamic_sleeve_target_weights,
)

In [14]:
gmv_sleeve_allocations = (
    gmv_allocations.set_index("date")[
        [
            "momentum_weight",
            "realised_volatility_weight",
        ]
    ]
    .rename(
        columns={
            "momentum_weight": MOMENTUM_SLEEVE,
            "realised_volatility_weight": VOLATILITY_SLEEVE,
        }
    )
    .sort_index()
)

gmv_sleeve_allocations.index = pd.to_datetime(gmv_sleeve_allocations.index)
gmv_sleeve_allocations.index.name = "date"

if not np.allclose(
    gmv_sleeve_allocations.sum(axis=1),
    1.0,
):
    raise ValueError("GMV sleeve allocations do not sum to one.")

allocation_dates = pd.DatetimeIndex(
    gmv_sleeve_allocations.index
).sort_values()

# Keep only target dates for which a GMV allocation exists.
gmv_input_sleeve_targets = {}

for sleeve_name in SLEEVE_NAMES:
    targets = sleeve_targets[sleeve_name].copy()

    targets["date"] = pd.to_datetime(
        targets["date"]
    )

    targets = (
        targets.loc[
            targets["date"].isin(allocation_dates)
        ]
        .sort_values(["date", "ticker"])
        .reset_index(drop=True)
    )

    gmv_input_sleeve_targets[sleeve_name] = targets

for sleeve_name, targets in gmv_input_sleeve_targets.items():
    target_dates = pd.DatetimeIndex(targets["date"].unique()).sort_values()

    missing_target_dates = allocation_dates.difference(target_dates)

    extra_target_dates = target_dates.difference(allocation_dates)

    if not missing_target_dates.empty:
        raise ValueError(
            f"{sleeve_name} is missing GMV allocation dates: "
            f"{missing_target_dates.tolist()}"
        )

    if not extra_target_dates.empty:
        raise ValueError(
            f"{sleeve_name} contains dates without GMV "
            f"allocations: {extra_target_dates.tolist()}"
        )

print("Aligned allocation dates:", len(allocation_dates))
print(
    "First allocation date:",
    allocation_dates.min().date(),
)
print(
    "Last allocation date:",
    allocation_dates.max().date(),
)

Aligned allocation dates: 501
First allocation date: 2016-07-14
Last allocation date: 2026-06-25


In [15]:
gmv_targets = (
    combine_dynamic_sleeve_target_weights(
        sleeve_targets=gmv_input_sleeve_targets,
        sleeve_allocations=gmv_sleeve_allocations,
    )
)

gmv_targets["date"] = pd.to_datetime(gmv_targets["date"])

gmv_target_dates = pd.DatetimeIndex(
    gmv_targets["date"].unique()
).sort_values()

if not gmv_target_dates.equals(allocation_dates):
    raise ValueError(
        "GMV target dates do not match "
        "the allocation dates."
    )

print("Allocation dates:", len(allocation_dates))
print("Target dates:", len(gmv_target_dates))
print("Target rows:", len(gmv_targets))

display(gmv_targets.head())

Allocation dates: 501
Target dates: 501
Target rows: 49359


,date,ticker,weight
0,2016-07-14,AAPL,-0.029051
1,2016-07-14,ABBV,-0.029051
2,2016-07-14,ABT,-0.029051
3,2016-07-14,ACN,0.029051
4,2016-07-14,ADBE,0.029051


In [16]:
gmv_daily, gmv_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=gmv_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=(TRANSACTION_COST_BPS),
)

gmv_daily["date"] = pd.to_datetime(gmv_daily["date"])

gmv_active = gmv_daily["gross_exposure"] > 1e-12

if not gmv_active.any():
    raise ValueError("The GMV portfolio never becomes active.")

gmv_active_start = gmv_daily.loc[
    gmv_active,
    "date",
].min()

gmv_active_end = gmv_daily.loc[
    gmv_active,
    "date",
].max()

if gmv_active_start != allocation_dates.min():
    raise ValueError("The GMV backtest does not begin on " "the first allocation date.")

print("Active start:", gmv_active_start.date())
print("Active end:", gmv_active_end.date())
print("Daily observations:", len(gmv_daily))
print("Holding rows:", len(gmv_holdings))

Active start: 2016-07-14
Active end: 2026-07-01
Daily observations: 2890
Holding rows: 284148


In [17]:
comparison_portfolios = {
    **benchmark_portfolios,
    "GMV Sleeves": gmv_daily,
}

common_evaluation_start = max(
    gmv_active_start,
    max(daily["date"].min() for daily in comparison_portfolios.values()),
)

common_evaluation_end = min(
    daily["date"].max() for daily in comparison_portfolios.values()
)

comparison_rows = []

for portfolio_name, daily in comparison_portfolios.items():
    evaluation_daily = daily.loc[
        daily["date"].between(
            common_evaluation_start,
            common_evaluation_end,
        )
    ].copy()

    gross_summary = summarise_backtest(
        evaluation_daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        evaluation_daily,
        return_column="net_return",
    ).iloc[0]

    comparison_rows.append(
        {
            "portfolio": portfolio_name,
            "start_date": evaluation_daily["date"].min(),
            "end_date": evaluation_daily["date"].max(),
            "observations": len(evaluation_daily),
            "gross_annualised_return": gross_summary["annualised_return"],
            "net_annualised_return": net_summary["annualised_return"],
            "net_annualised_volatility": net_summary["annualised_volatility"],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "total_transaction_cost": net_summary["total_transaction_cost"],
            "average_daily_gross_exposure": evaluation_daily["gross_exposure"].mean(),
        }
    )

gmv_comparison_summary = pd.DataFrame(comparison_rows).set_index("portfolio")

print(
    "Common evaluation period:",
    common_evaluation_start.date(),
    "to",
    common_evaluation_end.date(),
)

display(gmv_comparison_summary.round(4))

Common evaluation period: 2016-07-14 to 2026-07-01


C:\Users\39521\AppData\Local\Temp\ipykernel_12232\191184438.py:62: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(gmv_comparison_summary.round(4))


,start_date,end_date,observations,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,,,,
Composite Score,2016-07-14,2026-07-01,2505,0.1794,0.1489,0.2161,0.8723,0.7511,-0.3063,0.5194,0.2602,2.0006
Fixed 50/50 Sleeves,2016-07-14,2026-07-01,2505,0.1382,0.1135,0.1707,0.8441,0.7156,-0.2566,0.4345,0.2177,1.5519
Pure Inverse Volatility,2016-07-14,2026-07-01,2505,0.1404,0.1142,0.1650,0.8792,0.7381,-0.1957,0.4611,0.2310,1.6198
SPY Buy and Hold,2016-07-14,2026-07-01,2505,0.1505,0.1505,0.1795,0.8713,0.8713,-0.3372,NaN,0.0000,1.0000
Shrunk Inverse Volatility,2016-07-14,2026-07-01,2505,0.1397,0.1142,0.1662,0.8709,0.7345,-0.2103,0.4491,0.2250,1.5926
GMV Sleeves,2016-07-14,2026-07-01,2505,0.1148,0.0883,0.1701,0.7248,0.5827,-0.2077,0.4791,0.2400,1.6946


### GMV portfolio performance

The unconstrained covariance information did not improve the portfolio out of sample. Although the GMV portfolio was designed to minimise variance, its realised net volatility of 17.0% exceeded that of both the pure and shrunk inverse-volatility portfolios.

The weakness is already visible before transaction costs: GMV produced a gross Sharpe ratio of 0.72, compared with 0.88 for pure inverse volatility. Its time-varying allocations also increased turnover and reduced the net Sharpe ratio to 0.58.

Therefore, the raw GMV allocation is not retained as a preferred portfolio. The result illustrates the estimation risk of covariance optimisation. In particular, the strongly time-varying sleeve-correlation estimate creates substantial allocation variation without delivering a corresponding reduction in realised risk.

The next experiment regularises the GMV allocation by shrinking it toward a fixed 50/50 sleeve allocation.

### 1.1. Regularise the GMV weights

Use a fixed 50% shrinkage strength:

$$
w_t^{\mathrm{shrunk}}
=
(1-\lambda)w_t^{\mathrm{GMV}}
+\lambda
\begin{pmatrix} 0.5 \\ 0.5 \end{pmatrix},
\qquad \lambda=0.5.
$$

In [18]:
GMV_WEIGHT_SHRINKAGE = 0.50

equal_weight_anchor = pd.Series(
    {
        MOMENTUM_SLEEVE: 0.50,
        VOLATILITY_SLEEVE: 0.50,
    }
)

shrunk_gmv_sleeve_allocations = (
    (1.0 - GMV_WEIGHT_SHRINKAGE)
    * gmv_sleeve_allocations
    + GMV_WEIGHT_SHRINKAGE
    * equal_weight_anchor
)

shrunk_gmv_sleeve_allocations.index.name = "date"

if not np.allclose(
    shrunk_gmv_sleeve_allocations.sum(axis=1),
    1.0,
):
    raise ValueError(
        "Shrunk GMV allocations do not sum to one."
    )

if not shrunk_gmv_sleeve_allocations.ge(0.0).all().all():
    raise ValueError(
        "Shrunk GMV allocations contain negative weights."
    )

shrunk_gmv_weight_comparison = pd.DataFrame(
    {
        "raw_gmv_momentum":
            gmv_sleeve_allocations[MOMENTUM_SLEEVE],
        "shrunk_gmv_momentum":
            shrunk_gmv_sleeve_allocations[MOMENTUM_SLEEVE],
    }
)

display(shrunk_gmv_weight_comparison.describe().T)

,count,mean,std,min,25%,50%,75%,max
raw_gmv_momentum,501.0,0.5428,0.191067,0.00,0.46524,0.564023,0.695123,0.911079
shrunk_gmv_momentum,501.0,0.5214,0.095533,0.25,0.48262,0.532012,0.597562,0.705540


In [19]:
shrunk_gmv_targets = (
    combine_dynamic_sleeve_target_weights(
        sleeve_targets=gmv_input_sleeve_targets,
        sleeve_allocations=(
            shrunk_gmv_sleeve_allocations
        ),
    )
)

shrunk_gmv_targets["date"] = pd.to_datetime(
    shrunk_gmv_targets["date"]
)

shrunk_gmv_target_dates = pd.DatetimeIndex(
    shrunk_gmv_targets["date"].unique()
).sort_values()

if not shrunk_gmv_target_dates.equals(
    allocation_dates
):
    raise ValueError(
        "Shrunk GMV target dates do not match "
        "the allocation dates."
    )

shrunk_gmv_daily, shrunk_gmv_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=shrunk_gmv_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=TRANSACTION_COST_BPS,
    )
)

shrunk_gmv_daily["date"] = pd.to_datetime(
    shrunk_gmv_daily["date"]
)

print("Target rows:", len(shrunk_gmv_targets))
print("Daily observations:", len(shrunk_gmv_daily))
print("Holding rows:", len(shrunk_gmv_holdings))

Target rows: 49359
Daily observations: 2890
Holding rows: 284148


In [21]:
comparison_portfolios = {
    **benchmark_portfolios,
    "GMV Sleeves": gmv_daily,
    "Shrunk GMV Sleeves": shrunk_gmv_daily,
}

common_evaluation_start = max(
    gmv_active_start,
    max(daily["date"].min() for daily in comparison_portfolios.values()),
)

common_evaluation_end = min(
    daily["date"].max() for daily in comparison_portfolios.values()
)

comparison_rows = []

for portfolio_name, daily in comparison_portfolios.items():
    evaluation_daily = daily.loc[
        daily["date"].between(
            common_evaluation_start,
            common_evaluation_end,
        )
    ].copy()

    gross_summary = summarise_backtest(
        evaluation_daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        evaluation_daily,
        return_column="net_return",
    ).iloc[0]

    comparison_rows.append(
        {
            "portfolio": portfolio_name,
            "start_date": evaluation_daily["date"].min(),
            "end_date": evaluation_daily["date"].max(),
            "observations": len(evaluation_daily),
            "gross_annualised_return": gross_summary["annualised_return"],
            "net_annualised_return": net_summary["annualised_return"],
            "net_annualised_volatility": net_summary["annualised_volatility"],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "total_transaction_cost": net_summary["total_transaction_cost"],
            "average_daily_gross_exposure": evaluation_daily["gross_exposure"].mean(),
        }
    )

gmv_regularised_comparison_summary = pd.DataFrame(comparison_rows).set_index("portfolio")

print(
    "Common evaluation period:",
    common_evaluation_start.date(),
    "to",
    common_evaluation_end.date(),
)

display(gmv_regularised_comparison_summary.round(4))

Common evaluation period: 2016-07-14 to 2026-07-01


C:\Users\39521\AppData\Local\Temp\ipykernel_12232\727901557.py:63: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(gmv_regularised_comparison_summary.round(4))


,start_date,end_date,observations,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,,,,
Composite Score,2016-07-14,2026-07-01,2505,0.1794,0.1489,0.2161,0.8723,0.7511,-0.3063,0.5194,0.2602,2.0006
Fixed 50/50 Sleeves,2016-07-14,2026-07-01,2505,0.1382,0.1135,0.1707,0.8441,0.7156,-0.2566,0.4345,0.2177,1.5519
Pure Inverse Volatility,2016-07-14,2026-07-01,2505,0.1404,0.1142,0.1650,0.8792,0.7381,-0.1957,0.4611,0.2310,1.6198
SPY Buy and Hold,2016-07-14,2026-07-01,2505,0.1505,0.1505,0.1795,0.8713,0.8713,-0.3372,NaN,0.0000,1.0000
Shrunk Inverse Volatility,2016-07-14,2026-07-01,2505,0.1397,0.1142,0.1662,0.8709,0.7345,-0.2103,0.4491,0.2250,1.5926
GMV Sleeves,2016-07-14,2026-07-01,2505,0.1148,0.0883,0.1701,0.7248,0.5827,-0.2077,0.4791,0.2400,1.6946
Shrunk GMV Sleeves,2016-07-14,2026-07-01,2505,0.1271,0.1015,0.1673,0.7994,0.6620,-0.2137,0.4556,0.2282,1.6232


### GMV experiment summary

The first optimisation experiment applied a walk-forward global minimum-variance allocation to the momentum and realised-volatility sleeves. At each rebalance date, the covariance matrix was estimated using only prior returns, with a rolling window of up to 252 observations.

The raw GMV allocations were economically plausible. The optimiser assigned an average weight of 54.3% to momentum and 45.7% to realised volatility, while only approximately 1% of allocations reached the long-only boundary. However, the estimated sleeve correlation varied substantially over time, producing unstable allocations without a corresponding reduction in realised risk.

Over the common evaluation period, raw GMV achieved:

- net annualised return of 8.8%;
- net annualised volatility of 17.0%;
- net Sharpe ratio of 0.58; and
- maximum drawdown of 20.8%.

It did not improve on pure inverse volatility, which achieved lower volatility of 16.5% and a higher net Sharpe ratio of 0.74.

Shrinking the GMV allocation by 50% toward equal weights improved stability. The shrunk version raised the net Sharpe ratio to 0.66 and reduced volatility to 16.7%, but it still underperformed the simpler inverse-volatility methods.

The experiment therefore demonstrates that directly using the rolling correlation estimate adds estimation noise without improving portfolio performance. Raw and shrunk GMV will remain as comparison portfolios, but no further parameter tuning is pursued. The notebook now proceeds to alternative optimisation strategies.